[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/projection/projection.ipynb)

# Cameras and Shadows by Join and Meet

A pinhole camera and a shadow are the same construction. To find where a point appears in a camera, draw the line from the point through the camera's centre and see where it crosses the screen; to find where a point's shadow falls, draw the line from the light through the point and see where it crosses the ground. A sun far away is a light at infinity, and its shadows fall along one direction. Two cameras looking at the same scene are tied together the same way: a point seen by one camera must appear in the other somewhere along one line, its epipolar line.

In projective geometric algebra the line through two points is their join, `&`, and the point where a line crosses a plane is their meet, `^`. The camera is the map `(centre & Point) ^ screen`, a map from points to points; nothing in it cares whether the centre is a finite point or a direction at infinity, so the sun's shadow is the same expression. The same join and meet, as a map on the light, `(Point & corner) ^ ground`, gives a corner's shadow for every light position at once; as a map on lines it is the camera for lines, which images camera 1's sight rays as camera 2's epipolar lines; and as a map on two image points, `(centre_1 & Point) ^ (centre_2 & Point)`, it says whether their sight rays meet.

In the matrix notation of projective vision the camera reads as a $3 \times 4$ projection matrix $P$, the shadow maps as $4 \times 4$ shadow projection matrices, and the two-view correspondence form as the fundamental matrix $F$, with $\mathbf x_2^{\mathsf T} F \mathbf x_1 = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure
from examples.geometry.projection import core, render
from examples.geometry.projection.core import CUBE_EDGES, circle, cube

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                  # Antivector-
Line = ga.gatype.antibivector()                 # Antibivector-
Plane = ga.gatype.vector()                      # Vector+
Motor = ga.gatype.rotor()                       # Rotor+

## 1. The camera as a map

A camera at the origin looking along z, with its screen at `z == 1`, is the map `(origin & Point) ^ screen`. Its coefficient table shows the perspective division: the image's weight is the point's depth.

In [ ]:
origin = mv.w.dual()                                                           # [] Point-
screen = mv.z - mv.w                                                           # [] Plane+
camera = (origin & Point) ^ screen                                             # [] Point- <- Point-

In [ ]:
print("the camera, a map from points to their images:\n", camera.coefficient_table())

## 2. Shadows

A cube hangs above the ground, lit by a point light and by the sun. Each light's shadow map is the camera with the light as its centre and the ground as its screen. As a map on the light, the same expression gives one corner's shadow for every light position along a circle at once.

In [ ]:
# The cube's side and height above the ground; the point light, the sun's direction, and the circle of light
# positions, its centre and its count.
body_side, body_height = 1.0, 1.5
point_light, sun = mv.point([1, -1, 4]), mv.direction([-1, 0.6, -2.5])
light_path_centre, n_light_path = np.array([0, -1, 4]), 24

In [ ]:
body = (mv.zw * (body_height / 2)).exp() >> cube(body_side)                    # [corners] Point-
ground = mv.z                                                                  # [] Plane+
path_shift = ((stack((mv.xw, mv.yw, mv.zw)) * light_path_centre).sum(axis=0) / 2).exp()  # [] Motor+
light_path = path_shift >> circle(n_light_path)                                # [lights] Point-

# Join the point with the light into a ray, then meet the ray with the ground. As a map on
# the point this is linear, Point <- Point. Nothing in the expression cares whether the
# light is a finite point or a direction at infinity, so the sun's orthographic shadow is
# the same line of code.
spotlight = (point_light & Point) ^ ground                                     # [] Point- <- Point-
sunlight = (sun & Point) ^ ground                                              # [] Point- <- Point-
# With the corner fixed, its shadow is a linear map of the light's position.
shadow_of_corner = (Point & body[7]) ^ ground                                  # [] Point- <- Point-

point_shadow = spotlight(body)                                                 # [corners] Point-
sun_shadow = sunlight(body)                                                    # [corners] Point-
shadow_trail = shadow_of_corner(light_path)                                    # [lights] Point-
cast = point_shadow, sun_shadow, shadow_trail

## 3. Two cameras

The same camera is posed twice, shifted apart and turned toward each other, over a second cube. Each pose moves the camera as a map: world points are pulled into the rig's frame and images pushed back out. Camera 2's epipole is its image of camera 1's centre, and the epipolar line of each of camera 1's images is camera 2's image of its sight ray. In the figure, the left panel shows the blue cube, its yellow shadow under the point light, its purple shadow under the sun, and the beaded shadow trail of one corner as the light circles; the middle and right panels show the two cameras' images, with camera 2's corners lying on their epipolar lines.

In [ ]:
# Each rig's shift along x and turn toward the other, and the subject's side and depth in front of them.
rig_shift, rig_turn = 0.3, 0.06
subject_side, subject_depth = 1.6, 5.0

In [ ]:
rig_1 = (mv.xw * -rig_shift).exp() * (mv.xz * +rig_turn).exp()                # [] Motor+
rig_2 = (mv.xw * +rig_shift).exp() * (mv.xz * -rig_turn).exp()                # [] Motor+
subject = (mv.zw * (subject_depth / 2)).exp() >> cube(subject_side)            # [corners] Point-

# The same join then meet, as a map on lines, is the camera for lines.
line_camera = (origin & Line) ^ screen                                         # [] Line- <- Line-

# The camera moves as a map: pull world points back into the rig frame, push image
# points forward. The line camera moves the same way and stays consistent with the
# point camera: the image of a join is the join of the images.
camera_1 = rig_1 >> camera(rig_1 << Point)                                     # [] Point- <- Point-
camera_2 = rig_2 >> camera(rig_2 << Point)                                     # [] Point- <- Point-
line_camera_2 = rig_2 >> line_camera(rig_2 << Line)                            # [] Line- <- Line-
centre_1 = rig_1 >> origin                                                     # [] Point-
centre_2 = rig_2 >> origin                                                     # [] Point-

image_1 = camera_1(subject)                                                    # [corners] Point-
image_2 = camera_2(subject)                                                    # [corners] Point-

# The epipole is the image of the other centre. The epipolar line of an image point is
# the image of its ray, which the line camera gives directly.
epipole_2 = camera_2(centre_1)                                                 # [] Point-
epipolar_lines_2 = line_camera_2(centre_1 & image_1)                           # [corners] Line-

# Two image points correspond iff their rays meet, and two lines meet iff their wedge
# vanishes. As a map on both image points, that sentence is a bilinear form.
correspondence = (centre_1 & Point) ^ (centre_2 & Point)                       # [] Pseudoscalar+ <- (Point-, Point-)

views = image_1, image_2, epipole_2, epipolar_lines_2, correspondence
save_figure(render.draw_projection(body, CUBE_EDGES, cast, point_light, sun, views, rig_1, rig_2), "projection");

In [ ]:
# checks
# The shadows and stereo pair laid out here match the core's.
for built, reference in zip(cast, core.shadows(body, ground, point_light, sun, body[7], light_path)):
    np.testing.assert_allclose((built - reference).kernel, 0.0, atol=1e-12)
for built, reference in zip(views, core.stereo(subject, origin, screen, rig_1, rig_2)):
    np.testing.assert_allclose((built - reference).kernel, 0.0, atol=1e-12)
# Every shadow lies on the ground; corresponding images annihilate the correspondence form, the epipole lies in
# its kernel, and each epipolar line passes through its image.
for shadow in cast:
    assert np.abs((ground & shadow).to_array()).max() < 1e-12
assert np.abs(correspondence(image_1, image_2).dual().to_array()).max() < 1e-12
assert np.abs(correspondence.bind({1: epipole_2}).kernel).max() < 1e-12
assert (epipolar_lines_2 & image_2).norm().to_array().max() < 1e-12